# Tema 6 · Caso + Práctica · Procesamiento de Datos en Tiempo Real
## Monitoreo inteligente de tráfico con Spark Structured Streaming (simulación)

**Uso sugerido:** Para comprender cómo el procesamiento de datos en tiempo real permite optimizar decisiones operativas, ejecuta este cuaderno que simula un flujo de tráfico urbano y lo procesa con **Apache Spark Structured Streaming**.

**Contexto:** *CiudadMX* desea reducir congestiones adaptando semáforos según el flujo vehicular. Sensores (IoT) envían velocidad, conteos de autos y estado de semáforos. Aquí simulamos ese flujo y calculamos métricas en vivo.

## 1) Preparación del entorno
Instalamos **OpenJDK** y **PySpark** y creamos la sesión de Spark. Ejecuta esta celda primero.

In [ ]:
!apt-get -y install openjdk-17-jdk > /dev/null
!pip install -q pyspark==3.5.1
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Streaming_Trafico_CiudadMX").getOrCreate()
spark

## 2) Simulación de flujo de datos (micro-lotes)
En producción usaríamos **Kafka**; aquí generamos archivos CSV cada pocos segundos para simular eventos de sensores. Cada archivo es un "micro-lote" con varios cruces viales.

In [ ]:
import pandas as pd, numpy as np, time, random

def generar_eventos(n=10):
    calles = ["Av. Reforma", "Av. Juárez", "Av. Vallarta", "Av. Hidalgo"]
    datos = []
    for _ in range(n):
        evento = {
            "timestamp": pd.Timestamp.now(),
            "calle": random.choice(calles),
            "velocidad": np.random.randint(10, 90),
            "autos": np.random.randint(5, 40),
            "semaforo": random.choice(["Verde","Amarillo","Rojo"])
        }
        datos.append(evento)
    return pd.DataFrame(datos)

# Generamos 5 micro-lotes (archivos) con 2 s de separación
for i in range(5):
    df = generar_eventos(n=12)
    df.to_csv(f"/content/trafico_{i}.csv", index=False)
    time.sleep(2)
print("✅ Datos simulados generados en /content/")

## 3) Lectura continua (Structured Streaming)
Spark puede tratar un directorio como fuente de streaming y reaccionar a **nuevos archivos** que van apareciendo. Definimos el esquema y configuramos la lectura continua desde `/content/`.

In [ ]:
schema = "timestamp TIMESTAMP, calle STRING, velocidad INT, autos INT, semaforo STRING"
stream_df = spark.readStream.option("header", True).schema(schema).csv("/content/")
stream_df.printSchema()

## 4) Procesamiento en tiempo real
Calculamos métricas agregadas por calle: **velocidad promedio** y **autos totales**. Esto se actualiza en cada micro-lote de llegada de datos.

In [ ]:
from pyspark.sql import functions as F

resultados = (
    stream_df
    .groupBy("calle")
    .agg(
        F.avg("velocidad").alias("velocidad_promedio"),
        F.sum("autos").alias("autos_totales")
    )
)
resultados

## 5) Visualización en consola (salida continua)
Mostramos la tabla agregada cada 5 s. La consulta corre ~20 s y se detiene automáticamente. Vuelve a ejecutar la celda 2 para generar más micro-lotes si deseas observar más actualizaciones.

In [ ]:
consulta = (
    resultados.writeStream
    .outputMode("complete")
    .format("console")
    .trigger(processingTime="5 seconds")
    .start()
)
consulta.awaitTermination(20)
consulta.stop()

## 6) Interpretación de resultados
- **Velocidad baja + muchos autos** → posible congestión en esa calle.
- Cambios bruscos entre micro-lotes pueden activar **alertas** o ajustar la fase de semáforo.
- Este flujo puede integrarse con **Kafka** (ingesta), **MongoDB/HDFS** (persistencia) y **Power BI/Grafana** (dashboard).

## 7) Reflexión
1. ¿Por qué el *streaming* (micro-lotes) es más adecuado que el *batch* para control de tráfico?
2. ¿Cómo escalarías esta solución para miles de sensores?
3. ¿Qué métricas adicionales añadirías (p. ej., densidad, índice de congestión, tiempo medio de espera)?
4. ¿Qué limitaciones tiene esta simulación vs. un entorno con **Kafka** real?

## Recurso didáctico (libre y vigente)
**Confluent. (2024).** *Real-time streaming analytics with Apache Kafka.* Recuperado de https://www.confluent.io/resources/white-paper/real-time-streaming-analytics-with-apache-kafka/

> Este recurso explica patrones y buenas prácticas para *stream processing* con Kafka que puedes extrapolar a Spark Structured Streaming.

In [ ]:
# (Opcional) Cerrar sesión de Spark al final
spark.stop()
print("✅ Spark detenido")